In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.data.loader import load_hr_analytics
from src.models.baseline import get_logistic_regression_pipeline

# Load data
ds = load_hr_analytics("../data/raw/HR_Analytics.csv")

# Train baseline model
pipeline = get_logistic_regression_pipeline(ds.X)
pipeline.fit(ds.X, ds.y)

preprocessor = pipeline.named_steps["preprocess"]
model = pipeline.named_steps["model"]

# Encode features
X_processed = preprocessor.transform(ds.X).toarray()

# Compute baseline SHAP
explainer_baseline = shap.Explainer(
    model,
    X_processed,
    feature_names=preprocessor.get_feature_names_out()
)

shap_vals_baseline = explainer_baseline(X_processed)

# baseline SHAP
shap_values = shap_vals_baseline   # from Monday notebook
X_enc = X_processed                # same encoded features
A = ds.A.reset_index(drop=True)    # Gender series
feature_names = preprocessor.get_feature_names_out()

male_idx = A[A == "Male"].index
female_idx = A[A == "Female"].index

shap_male = shap_values.values[male_idx]
shap_female = shap_values.values[female_idx]

male_importance = np.abs(shap_male).mean(axis=0)
female_importance = np.abs(shap_female).mean(axis=0)

df_imp = pd.DataFrame({
    "feature": feature_names,
    "Male": male_importance,
    "Female": female_importance,
})

df_imp["diff"] = df_imp["Male"] - df_imp["Female"]

top_male = df_imp.sort_values("Male", ascending=False).head(5)
top_female = df_imp.sort_values("Female", ascending=False).head(5)

out = Path("../runs/shap_subgroups")
out.mkdir(parents=True, exist_ok=True)

top = df_imp.sort_values("Male", ascending=False).head(5)

x = np.arange(len(top))
width = 0.35


plt.figure(figsize=(8,5))
plt.bar(x - width/2, top["Male"], width, label="Male")
plt.bar(x + width/2, top["Female"], width, label="Female")

plt.xticks(x, top["feature"], rotation=45, ha="right")
plt.ylabel("Mean |SHAP|")
plt.title("Subgroup Feature Importance (Baseline)")
plt.legend()

plt.tight_layout()
plt.savefig(out / "shap_subgroup_baseline.png", dpi=300)
plt.close()

top_diff = df_imp.sort_values("diff", key=abs, ascending=False).head(5)

plt.figure(figsize=(7,4))
plt.barh(top_diff["feature"], top_diff["diff"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Male − Female Mean |SHAP|")
plt.title("Subgroup SHAP Differences")

plt.tight_layout()
plt.savefig(out / "shap_subgroup_diff_baseline.png", dpi=300)
plt.close()

from src.mitigation.exponentiated_gradient import train_expgrad
from src.data.preprocessing import build_preprocessor

pre_eg = build_preprocessor(ds.X)
X_enc_eg = pre_eg.fit_transform(ds.X).toarray()

eg_dp = train_expgrad(
    X_enc_eg,
    ds.y,
    ds.A,
    constraint="dp",
    seed=42,
)


eg_predictor = eg_dp.predictors_[0]


explainer_eg = shap.Explainer(
    eg_predictor,
    X_enc_eg,
    feature_names=pre_eg.get_feature_names_out()
)

shap_vals_eg = explainer_eg(X_enc_eg)

male_idx = A[A == "Male"].index
female_idx = A[A == "Female"].index

shap_male_eg = shap_vals_eg.values[male_idx]
shap_female_eg = shap_vals_eg.values[female_idx]

male_imp_eg = np.abs(shap_male_eg).mean(axis=0)
female_imp_eg = np.abs(shap_female_eg).mean(axis=0)

df_eg = pd.DataFrame({
    "feature": pre_eg.get_feature_names_out(),
    "Male": male_imp_eg,
    "Female": female_imp_eg,
})

df_eg["diff"] = df_eg["Male"] - df_eg["Female"]

top_eg = df_eg.sort_values("Male", ascending=False).head(5)

x = np.arange(len(top_eg))
width = 0.35

plt.figure(figsize=(8,5))
plt.bar(x - width/2, top_eg["Male"], width, label="Male")
plt.bar(x + width/2, top_eg["Female"], width, label="Female")

plt.xticks(x, top_eg["feature"], rotation=45, ha="right")
plt.ylabel("Mean |SHAP|")
plt.title("Subgroup Feature Importance (Exponentiated Gradient – DP)")
plt.legend()

plt.tight_layout()
plt.savefig(out / "shap_subgroup_expgrad_dp.png", dpi=300)
plt.close()

top_diff_eg = df_eg.sort_values(
    "diff", key=abs, ascending=False
).head(5)

plt.figure(figsize=(7,4))
plt.barh(top_diff_eg["feature"], top_diff_eg["diff"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Male − Female Mean |SHAP|")
plt.title("Subgroup SHAP Differences (Exponentiated Gradient – DP)")

plt.tight_layout()
plt.savefig(out / "shap_subgroup_diff_expgrad_dp.png", dpi=300)
plt.close()

